In [1]:
import subprocess, sys

# Step 1 — main packages
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "ultralytics", "opencv-python", "torch", "torchvision",
    "pytorchvideo", "fvcore", "iopath", "numpy<2",
    "timm", "boto3", "requests",
    "--quiet"
])

# Step 2 — mediapipe separately to avoid protobuf conflict
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "mediapipe==0.10.5",
    "--quiet"
])

print("Done.")

Done.


In [2]:
import cv2, os, csv, torch, json, urllib.request
import numpy as np
import boto3
import requests
import threading
import time
from ultralytics import YOLO
from datetime import datetime
from collections import deque
from botocore.config import Config

# MediaPipe — import safely so other computers don't crash if it fails
try:
    import mediapipe as mp
    MEDIAPIPE_AVAILABLE = True
    print(f"MediaPipe version: {mp.__version__}")
except Exception as e:
    MEDIAPIPE_AVAILABLE = False
    print(f"MediaPipe not available: {e}")
    print("Will run without fingertip detection — wrists only.")

print("Imports OK.")

MediaPipe version: 0.10.5
Imports OK.


In [3]:
# ── Cloudflare R2 credentials ──────────────────────────────────────────────
CF_ACCOUNT_ID      = "e15cd4ea0c022be46bbe1c0afa742ead"
CF_ACCESS_KEY_ID   = "fd60c041f1a4ed8eeb39596ad0bb0449"
CF_SECRET_KEY      = "f422288a54c9c553455de119c6a4fc08663a3e3dbf56f6850888269e8db3a484"
CF_BUCKET_NAME     = "sfc"
ALERT_WEBHOOK_URL  = None  # add webhook URL if you have one e.g. Discord/Slack

CF_ENDPOINT_URL    = f"https://{CF_ACCOUNT_ID}.r2.cloudflarestorage.com"

r2_client = boto3.client(
    "s3",
    endpoint_url          = CF_ENDPOINT_URL,
    aws_access_key_id     = CF_ACCESS_KEY_ID,
    aws_secret_access_key = CF_SECRET_KEY,
    config                = Config(signature_version="s3v4"),
    region_name           = "auto"
)
print("Cloudflare R2 client ready.")

try:
    r2_client.head_bucket(Bucket=CF_BUCKET_NAME)
    print(f"Connected to bucket: {CF_BUCKET_NAME}")
except Exception as e:
    print(f"WARNING: R2 connection failed — {e}")
    print("Uploads will be skipped but local saving still works.")

# ── YOLOv8 Pose ────────────────────────────────────────────────────────────
pose_model = YOLO("yolov8m-pose.pt")
print("Pose model loaded.")

# ── YOLOv8 Object detection ────────────────────────────────────────────────
yolo_model = YOLO("yolov8m.pt")
print("YOLO object model loaded.")

# ── X3D Action model ───────────────────────────────────────────────────────
action_model = torch.hub.load(
    "facebookresearch/pytorchvideo",
    "x3d_m", pretrained=True
)
action_model.eval()
print("X3D action model loaded.")

# ── MiDaS depth estimation ─────────────────────────────────────────────────
midas            = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
midas_transform  = midas_transforms.small_transform
midas.eval()
print("MiDaS loaded.")

# ── MediaPipe Hands (optional — works without it) ──────────────────────────
USE_OLD_API   = False
hands         = None

if MEDIAPIPE_AVAILABLE:
    try:
        mp_hands = mp.solutions.hands
        hands    = mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )
        USE_OLD_API = True
        print("MediaPipe hands loaded (old API).")
    except AttributeError:
        try:
            model_path = "hand_landmarker.task"
            if not os.path.exists(model_path):
                print("Downloading hand landmarker model...")
                urllib.request.urlretrieve(
                    "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
                    model_path
                )
                print("Downloaded.")
            HandLandmarker        = mp.tasks.vision.HandLandmarker
            HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
            BaseOptions           = mp.tasks.BaseOptions
            RunningMode           = mp.tasks.vision.RunningMode
            options = HandLandmarkerOptions(
                base_options=BaseOptions(model_asset_path=model_path),
                running_mode=RunningMode.IMAGE,
                num_hands=2
            )
            hands       = HandLandmarker.create_from_options(options)
            USE_OLD_API = False
            print("MediaPipe hands loaded (Tasks API).")
        except Exception as e:
            print(f"MediaPipe hands failed to load: {e}")
            print("Running without fingertip detection.")
else:
    print("MediaPipe not available — running without fingertip detection.")

# ── Kinetics-400 labels ────────────────────────────────────────────────────
ACTION_MAP = {
    "running":  ["running", "jogging", "sprinting"],
    "walking":  ["walking", "marching"],
    "smoking":  ["smoking", "smoking hookah"],
    "shooting": ["shooting", "archery", "shooting goal"],
    "cutting":  ["cutting", "slicing", "chopping wood"],
    "throwing": ["throwing", "javelin throw", "shot put"],
    "standing": ["standing"],
    "plucking": ["picking fruit", "gardening", "pulling rope", "trimming trees"]
}

labels_url  = "https://dl.fbaipublicfiles.com/pyslowfast/dataset/class_names/kinetics_classnames.json"
labels_path = "kinetics_labels.json"
if not os.path.exists(labels_path):
    print("Downloading Kinetics labels...")
    urllib.request.urlretrieve(labels_url, labels_path)

with open(labels_path) as f:
    kinetics_labels = json.load(f)
id2label = {int(v): k for k, v in kinetics_labels.items()}
print(f"Loaded {len(id2label)} Kinetics labels.")

def map_to_action(kinetics_label):
    label_lower = kinetics_label.lower()
    for action, keywords in ACTION_MAP.items():
        for kw in keywords:
            if kw in label_lower:
                return action
    return "unknown"

c:\Users\Chai Kho Hua\anaconda3\lib\site-packages\boto3\compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


Cloudflare R2 client ready.
Connected to bucket: sfc
Pose model loaded.
YOLO object model loaded.


Using cache found in C:\Users\Chai Kho Hua/.cache\torch\hub\facebookresearch_pytorchvideo_main


X3D action model loaded.


Using cache found in C:\Users\Chai Kho Hua/.cache\torch\hub\intel-isl_MiDaS_master


Loading weights:  None


Using cache found in C:\Users\Chai Kho Hua/.cache\torch\hub\rwightman_gen-efficientnet-pytorch_master


MiDaS loaded.
MediaPipe hands loaded (old API).
Loaded 400 Kinetics labels.


Using cache found in C:\Users\Chai Kho Hua/.cache\torch\hub\intel-isl_MiDaS_master


In [4]:
# ── Tunable settings ───────────────────────────────────────────────────────
KEYPOINT_CONF    = 0.5
DEPTH_THRESHOLD  = 0.08
VIOLATION_FRAMES = 5
PROCESS_EVERY_N  = 2
RESIZE_W         = 640
RESIZE_H         = 480
CLIP_LEN         = 16

HIGH_SEVERITY   = {"cutting", "shooting", "throwing", "smoking", "plucking"}
MEDIUM_SEVERITY = {"running", "walking"}
LOW_SEVERITY    = {"standing", "unknown"}

ANIMAL_LABELS = {
    "dog", "cat", "horse", "cow", "elephant",
    "bear", "zebra", "giraffe", "bird", "sheep"
}

FINGERTIP_INDICES = [4, 8, 12, 16, 20]


def get_severity(action, target_type="plant"):
    if target_type == "animal":
        return "HIGH", (0, 0, 255)
    if action in HIGH_SEVERITY:   return "HIGH",   (0, 0, 255)
    if action in MEDIUM_SEVERITY: return "MEDIUM", (0, 165, 255)
    return "LOW", (0, 255, 255)


def upload_to_r2_and_alert(filepath, timestamp, action, severity,
                            violation_label, violation_type):
    """Runs in background thread — uploads clip to R2 and sends alert."""
    filename  = os.path.basename(filepath)
    r2_key    = f"violations/{datetime.now().strftime('%Y/%m/%d')}/{filename}"
    upload_url = None

    try:
        print(f"[R2] Uploading {filename}...")
        r2_client.upload_file(
            filepath, CF_BUCKET_NAME, r2_key,
            ExtraArgs={"ContentType": "video/mp4"}
        )
        upload_url = r2_client.generate_presigned_url(
            "get_object",
            Params={"Bucket": CF_BUCKET_NAME, "Key": r2_key},
            ExpiresIn=86400
        )
        print(f"[R2] Uploaded: {r2_key}")
        print(f"[R2] URL: {upload_url}")
    except Exception as e:
        print(f"[R2] Upload failed: {e}")

    alert_payload = {
        "timestamp"     : timestamp,
        "violation_type": violation_type,
        "action"        : action,
        "severity"      : severity,
        "description"   : violation_label,
        "clip_filename" : filename,
        "r2_key"        : r2_key,
        "clip_url"      : upload_url or "upload failed"
    }

    if ALERT_WEBHOOK_URL:
        try:
            message = (
                f"🚨 *VIOLATION DETECTED*\n"
                f"*Severity:* {severity}\n"
                f"*Type:* {violation_type}\n"
                f"*Action:* {action}\n"
                f"*Description:* {violation_label}\n"
                f"*Time:* {timestamp}\n"
                f"*Clip:* {upload_url or 'upload failed'}"
            )
            requests.post(ALERT_WEBHOOK_URL,
                          json={"text": message}, timeout=10)
            print("[ALERT] Webhook sent.")
        except Exception as e:
            print(f"[ALERT] Webhook failed: {e}")
    else:
        print(f"[ALERT] {alert_payload}")


def detect_plants_combined(frame, yolo_conf=0.4):
    hsv         = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask        = cv2.inRange(hsv,
                              np.array([35,  60,  30]),
                              np.array([85, 255, 200]))
    kernel      = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask        = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel, iterations=2)
    mask        = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    hsv_boxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 3000:
            continue
        x, y, bw, bh = cv2.boundingRect(cnt)
        if bw / max(bh, 1) > 3.0:
            continue
        hsv_boxes.append((x, y, x+bw, y+bh))

    if not hsv_boxes:
        return []

    PLANT_LABELS = {"potted plant", "plant", "tree", "vase"}
    results      = yolo_model(frame, verbose=False, conf=yolo_conf)
    yolo_boxes   = []
    for box, cls in zip(results[0].boxes.xyxy.cpu().numpy(),
                        results[0].boxes.cls.cpu().numpy()):
        if yolo_model.names[int(cls)].lower() in PLANT_LABELS:
            yolo_boxes.append(box)

    if not yolo_boxes:
        return []

    confirmed = []
    for (hx1, hy1, hx2, hy2) in hsv_boxes:
        for ybox in yolo_boxes:
            yx1, yy1, yx2, yy2 = ybox
            if hx1 < yx2 and hx2 > yx1 and hy1 < yy2 and hy2 > yy1:
                confirmed.append((hx1, hy1, hx2, hy2))
                break
    return confirmed


def detect_animals(frame, yolo_conf=0.4):
    results      = yolo_model(frame, verbose=False, conf=yolo_conf)
    animal_boxes = []
    for box, cls in zip(results[0].boxes.xyxy.cpu().numpy(),
                        results[0].boxes.cls.cpu().numpy()):
        label = yolo_model.names[int(cls)]
        if label.lower() in ANIMAL_LABELS:
            x1, y1, x2, y2 = box
            animal_boxes.append((int(x1), int(y1), int(x2), int(y2), label))
    return animal_boxes


def get_fingertips_mediapipe(frame):
    """
    Returns fingertip positions.
    Works with old API, Tasks API, or returns empty list if unavailable.
    Never crashes — always safe to call.
    """
    if not MEDIAPIPE_AVAILABLE or hands is None:
        return []

    try:
        fingertips = []
        h, w       = frame.shape[:2]
        rgb        = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        if USE_OLD_API:
            results = hands.process(rgb)
            if not results.multi_hand_landmarks:
                return []
            for hand_landmarks in results.multi_hand_landmarks:
                for idx in FINGERTIP_INDICES:
                    lm = hand_landmarks.landmark[idx]
                    fingertips.append((int(lm.x * w), int(lm.y * h)))
        else:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result   = hands.detect(mp_image)
            if not result.hand_landmarks:
                return []
            for hand in result.hand_landmarks:
                for idx in FINGERTIP_INDICES:
                    lm = hand[idx]
                    fingertips.append((int(lm.x * w), int(lm.y * h)))

        return fingertips
    except Exception:
        return []


def get_keypoints_and_boxes(frame):
    parts, person_boxes = [], []
    results = pose_model(cv2.resize(frame, (RESIZE_W, RESIZE_H)), verbose=False)
    h_orig, w_orig = frame.shape[:2]
    sx, sy = w_orig / RESIZE_W, h_orig / RESIZE_H

    person_height = 0
    for person in results:
        if person.keypoints is None or person.boxes is None:
            continue
        kps      = person.keypoints.xy.cpu().numpy()
        kps_conf = person.keypoints.conf.cpu().numpy()
        p_boxes  = person.boxes.xyxy.cpu().numpy()

        for i, (person_kps, person_conf) in enumerate(zip(kps, kps_conf)):
            if i < len(p_boxes):
                pb            = p_boxes[i]
                person_height = (pb[3] - pb[1]) * sy
                person_boxes.append((
                    int(pb[0]*sx), int(pb[1]*sy),
                    int(pb[2]*sx), int(pb[3]*sy)
                ))
            else:
                person_height = 0

            for idx, part_name in [(9,  "hand"), (10, "hand"),
                                   (15, "foot"), (16, "foot")]:
                if idx >= len(person_kps) or person_conf[idx] < KEYPOINT_CONF:
                    continue
                x, y = person_kps[idx]
                if x == 0 and y == 0:
                    continue
                parts.append((float(x*sx), float(y*sy), part_name, person_height))

    for (fx, fy) in get_fingertips_mediapipe(frame):
        parts.append((float(fx), float(fy), "fingertip", person_height))

    return parts, person_boxes


def estimate_depth(frame):
    img       = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    inp       = midas_transform(img).to("cpu")
    with torch.no_grad():
        pred  = midas(inp)
        depth = torch.nn.functional.interpolate(
            pred.unsqueeze(1),
            size=img.shape[:2],
            mode="bicubic",
            align_corners=False
        ).squeeze().cpu().numpy()
    d_min, d_max = depth.min(), depth.max()
    if d_max - d_min > 0:
        depth = (depth - d_min) / (d_max - d_min)
    return depth


def is_touching(kx, ky, target_box, depth_map):
    px1, py1, px2, py2 = target_box
    if not (px1 < kx < px2 and py1 < ky < py2):
        return False
    h, w          = depth_map.shape
    target_region = depth_map[max(0, int(py1)):min(h, int(py2)),
                               max(0, int(px1)):min(w, int(px2))]
    if target_region.size == 0:
        return False
    target_back_depth  = np.percentile(target_region, 20)
    target_front_depth = np.percentile(target_region, 80)
    kx_i     = int(np.clip(kx, 0, w - 1))
    ky_i     = int(np.clip(ky, 0, h - 1))
    kp_depth = depth_map[ky_i, kx_i]
    return target_back_depth <= kp_depth <= (target_front_depth + DEPTH_THRESHOLD)


def predict_action(frame_buffer):
    mean = [0.45, 0.45, 0.45]
    std  = [0.225, 0.225, 0.225]

    def preprocess(frame):
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (256, 256))
        img = img.astype(np.float64) / 255.0
        img = (img - mean) / std
        return torch.tensor(img).permute(2, 0, 1)

    frames_tensor = torch.stack([preprocess(f) for f in frame_buffer])
    inp           = frames_tensor.permute(1, 0, 2, 3).unsqueeze(0).double()
    with torch.no_grad():
        preds = action_model.double()(inp)
    top_label = id2label[preds.argmax().item()]
    return map_to_action(top_label)


print("Helper functions defined.")

Helper functions defined.


In [5]:
base_folder  = "bodycam"
raw_folder   = os.path.join(base_folder, "raw")
viol_folder  = os.path.join(base_folder, "violation")
os.makedirs(raw_folder,  exist_ok=True)
os.makedirs(viol_folder, exist_ok=True)

raw_out_path = os.path.join(raw_folder,
    f"bodycam_raw_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.mp4")

log_file = os.path.join(viol_folder, "violations_log.csv")
if not os.path.exists(log_file):
    with open(log_file, "w", newline="") as f:
        csv.writer(f).writerow(["timestamp", "action", "severity",
                                 "violation_type", "clip_filepath",
                                 "r2_key", "clip_url"])

print(f"Raw  : {raw_out_path}")
print(f"Clips: {viol_folder}")
print(f"Log  : {log_file}")

Raw  : bodycam\raw\bodycam_raw_2026-04-30_09-20-11.mp4
Clips: bodycam\violation
Log  : bodycam\violation\violations_log.csv


In [6]:
# ── Camera setup ───────────────────────────────────────────────────────────
LAPTOP_CAM_IDX = 0
PHONE_CAM_IDXS = [1, 2, 3, 4]


def is_valid_camera(cap):
    for _ in range(10):
        ret, frame = cap.read()
    if not ret or frame is None or frame.size == 0:
        return False
    if frame.shape[0] == 0 or frame.shape[1] == 0:
        return False
    b, g, r    = cv2.split(frame)
    frame_mean = frame.mean()
    frame_std  = frame.std()
    if g.mean() > 50 and g.mean() > b.mean() * 2 and g.mean() > r.mean() * 2:
        print("  → green screen rejected")
        return False
    if frame_mean < 10:
        print("  → black screen rejected")
        return False
    if 100 < frame_mean < 150 and frame_std < 50:
        print(f"  → Camo placeholder rejected (mean={frame_mean:.1f} std={frame_std:.1f})")
        return False
    if frame_std < 5:
        print("  → uniform/empty frame rejected")
        return False
    print(f"  → valid frame (mean={frame_mean:.1f} std={frame_std:.1f})")
    return True


def measure_fps(cap, n_frames=30):
    """Measures actual camera FPS by timing real frame reads."""
    frames    = []
    t_start   = time.time()
    for _ in range(n_frames):
        ret, f = cap.read()
        if ret and f is not None:
            frames.append(f)
    elapsed = time.time() - t_start
    fps     = len(frames) / elapsed if elapsed > 0 else 20.0
    return max(5.0, min(fps, 30.0))


# ── Try phone camera first ─────────────────────────────────────────────────
phone_cap = None
for idx in PHONE_CAM_IDXS:
    print(f"Checking index {idx}...")
    test = cv2.VideoCapture(idx, cv2.CAP_MSMF)
    if not test.isOpened():
        print(f"  → not available")
        test.release()
        continue
    test.set(cv2.CAP_PROP_FRAME_WIDTH,  1280)
    test.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    test.set(cv2.CAP_PROP_FPS, 30)
    if is_valid_camera(test):
        phone_cap = test
        print(f"  → Phone camera connected at index {idx}")
        break
    else:
        test.release()

if phone_cap is None:
    print("Phone camera not found — using laptop camera.")
    cap = cv2.VideoCapture(LAPTOP_CAM_IDX, cv2.CAP_MSMF)
    if not cap.isOpened():
        raise RuntimeError("Laptop camera not found.")
    print("Laptop camera connected.")
else:
    cap = phone_cap
    print("Phone camera connected.")

# ── Measure actual FPS ─────────────────────────────────────────────────────
print("Measuring actual FPS...")
actual_fps = measure_fps(cap)
print(f"Actual FPS measured: {actual_fps:.1f}")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps    = actual_fps
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Camera resolution: {width}x{height} @ {fps:.1f}fps")

if width == 0 or height == 0:
    raise RuntimeError("Camera returned 0x0.")

raw_out = cv2.VideoWriter(raw_out_path, fourcc, fps, (width, height))

# ── Verify stream is live ──────────────────────────────────────────────────
print("Verifying camera stream...")
stream_ok = False
for _ in range(30):
    ret, test_frame = cap.read()
    if ret and test_frame is not None and test_frame.mean() > 10:
        if not (100 < test_frame.mean() < 150 and test_frame.std() < 50):
            print(f"Camera stream verified — brightness: {test_frame.mean():.1f}")
            stream_ok = True
            break

if not stream_ok:
    print("WARNING: Phone stream invalid — switching to laptop camera.")
    cap.release()
    raw_out.release()
    cap = cv2.VideoCapture(LAPTOP_CAM_IDX, cv2.CAP_MSMF)
    print("Re-measuring FPS for laptop camera...")
    actual_fps = measure_fps(cap)
    fps        = actual_fps
    width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Laptop camera — resolution: {width}x{height} @ {fps:.1f}fps")
    raw_out = cv2.VideoWriter(raw_out_path, fourcc, fps, (width, height))

violation_writer      = None
violation_active      = False
clip_index            = 0
violation_frame_count = 0
frame_count           = 0
frame_buffer          = deque(maxlen=CLIP_LEN)
current_action        = "unknown"

last_plants       = []
last_animals      = []
last_parts        = []
last_person_boxes = []
last_all_targets  = []
last_depth_map    = None
touching_this_frame = False
violation_label     = ""
violation_type      = "plant"

fps_counter    = 0
fps_start_time = time.time()
real_fps       = fps

consecutive_failures = 0
print("Starting... Press 'q' to quit.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or frame is None or frame.size == 0:
        consecutive_failures += 1
        if consecutive_failures > 30:
            print("Camera stream lost — exiting.")
            break
        continue
    if frame.shape[0] == 0 or frame.shape[1] == 0:
        consecutive_failures += 1
        continue
    if frame.mean() < 5:
        consecutive_failures += 1
        continue
    consecutive_failures = 0

    raw_out.write(frame)
    frame_count += 1
    frame_buffer.append(frame.copy())

    # Update real FPS every 30 frames
    fps_counter += 1
    if fps_counter >= 30:
        real_fps       = fps_counter / (time.time() - fps_start_time)
        real_fps       = max(5.0, min(real_fps, 30.0))
        fps_counter    = 0
        fps_start_time = time.time()

    if frame_count % PROCESS_EVERY_N == 0:
        last_plants                   = detect_plants_combined(frame)
        last_animals                  = detect_animals(frame)
        last_parts, last_person_boxes = get_keypoints_and_boxes(frame)

        if len(frame_buffer) == CLIP_LEN and frame_count % CLIP_LEN == 0:
            current_action = predict_action(list(frame_buffer))
            print(f"Action detected: {current_action}")

        touching_this_frame = False
        violation_label     = ""
        violation_type      = "plant"

        last_all_targets = [(x1, y1, x2, y2, "plant")
                            for (x1, y1, x2, y2) in last_plants]
        last_all_targets += [(x1, y1, x2, y2, animal_label)
                             for (x1, y1, x2, y2, animal_label) in last_animals]

        any_inside = any(
            tx1 < kx < tx2 and ty1 < ky < ty2
            for (kx, ky, _, __) in last_parts
            for (tx1, ty1, tx2, ty2, _tl) in last_all_targets
        )

        last_depth_map = estimate_depth(frame) if any_inside else None

        if last_depth_map is not None:
            for (kx, ky, part, _) in last_parts:
                for (tx1, ty1, tx2, ty2, target_label) in last_all_targets:
                    if is_touching(kx, ky, (tx1, ty1, tx2, ty2), last_depth_map):
                        touching_this_frame = True
                        is_animal           = target_label in ANIMAL_LABELS
                        violation_type      = "animal" if is_animal else "plant"
                        violation_label     = (
                            f"{part.capitalize()} touching {target_label} "
                            f"while {current_action}"
                        )

    if touching_this_frame:
        violation_frame_count += 1
    else:
        violation_frame_count = 0

    confirmed = violation_frame_count >= VIOLATION_FRAMES

    # ── Draw ──────────────────────────────────────────────────────────────
    annotated   = frame.copy()
    part_colors = {
        "hand":      (255, 100,   0),
        "foot":      (0,   165, 255),
        "fingertip": (0,   255, 255),
    }

    for (bx1, by1, bx2, by2) in last_person_boxes:
        cv2.rectangle(annotated, (bx1, by1), (bx2, by2), (0, 215, 255), 2)
        cv2.putText(annotated, f"person: {current_action}",
                    (bx1, by1 - 5), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, (0, 215, 255), 1)

    for (px1, py1, px2, py2) in last_plants:
        cv2.rectangle(annotated, (px1, py1), (px2, py2), (0, 200, 0), 2)
        cv2.putText(annotated, "plant", (px1, py1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 200, 0), 1)

    for (ax1, ay1, ax2, ay2, animal_label) in last_animals:
        cv2.rectangle(annotated, (ax1, ay1), (ax2, ay2), (255, 0, 255), 2)
        cv2.putText(annotated, animal_label, (ax1, ay1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 255), 1)

    for (kx, ky, part, _) in last_parts:
        color  = part_colors.get(part, (255, 255, 255))
        radius = 4 if part == "fingertip" else 8
        cv2.circle(annotated, (int(kx), int(ky)), radius, color, -1)
        if part != "fingertip":
            cv2.putText(annotated, part, (int(kx) + 10, int(ky)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    # ── Depth debug overlay ────────────────────────────────────────────────
    if last_depth_map is not None:
        for (kx, ky, part, _) in last_parts:
            kx_i = int(np.clip(kx, 0, last_depth_map.shape[1] - 1))
            ky_i = int(np.clip(ky, 0, last_depth_map.shape[0] - 1))
            kp_d = last_depth_map[ky_i, kx_i]
            cv2.putText(annotated, f"{part}:{kp_d:.2f}",
                        (int(kx) + 10, int(ky) - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
        for (tx1, ty1, tx2, ty2, tlabel) in last_all_targets:
            region = last_depth_map[
                max(0, ty1):min(last_depth_map.shape[0], ty2),
                max(0, tx1):min(last_depth_map.shape[1], tx2)
            ]
            if region.size > 0:
                t_back  = np.percentile(region, 20)
                t_front = np.percentile(region, 80)
                cv2.putText(annotated,
                            f"{tlabel} depth:{t_back:.2f}-{t_front:.2f}",
                            (tx1, ty2 + 15),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

    # ── Violation ─────────────────────────────────────────────────────────
    if confirmed:
        severity, sev_color = get_severity(current_action, violation_type)
        if not violation_active:
            clip_index    += 1
            timestamp      = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            viol_out_path  = os.path.join(
                viol_folder,
                f"violation_{severity}_{violation_type}_{current_action}"
                f"_{clip_index}_{timestamp}.mp4"
            )
            violation_writer = cv2.VideoWriter(
                viol_out_path, fourcc, real_fps, (width, height)
            )
            violation_active = True
            print(f"[{severity} VIOLATION] {violation_label} at {timestamp}")

            with open(log_file, "a", newline="") as f:
                csv.writer(f).writerow([timestamp, current_action,
                                        severity, violation_label,
                                        viol_out_path, "", "pending"])

            threading.Thread(
                target=upload_to_r2_and_alert,
                args=(viol_out_path, timestamp, current_action,
                      severity, violation_label, violation_type),
                daemon=True
            ).start()

        cv2.putText(annotated, f"[{severity}] {violation_label}",
                    (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, sev_color, 2)
        cv2.rectangle(annotated, (0, 0), (width - 1, height - 1), sev_color, 4)
        if violation_writer is not None:
            violation_writer.write(annotated)

    elif violation_active:
        violation_writer.release()
        violation_writer = None
        violation_active = False
        print("[VIOLATION] Clip ended.")

    # ── Debug overlay ──────────────────────────────────────────────────────
    cv2.putText(annotated,
                f"Action: {current_action} | "
                f"Touch: {violation_frame_count}/{VIOLATION_FRAMES} | "
                f"FPS: {real_fps:.1f}",
                (10, height - 10), cv2.FONT_HERSHEY_SIMPLEX,
                0.5, (200, 200, 200), 1)

    cv2.imshow("Bodycam - Plant + Animal Touch Detection", annotated)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ── Cleanup ────────────────────────────────────────────────────────────────
cap.release()
raw_out.release()
if violation_writer is not None:
    violation_writer.release()
cv2.destroyAllWindows()

print(f"\nRaw video : {raw_out_path}")
print(f"Clips     : {viol_folder}")
print(f"Log       : {log_file}")q

Checking index 1...
  → valid frame (mean=18.5 std=8.4)
  → Phone camera connected at index 1
Phone camera connected.
Measuring actual FPS...
Actual FPS measured: 30.0
Camera resolution: 1280x720 @ 30.0fps
Verifying camera stream...
Re-measuring FPS for laptop camera...
Laptop camera — resolution: 640x480 @ 22.7fps
Starting... Press 'q' to quit.
Action detected: unknown
Action detected: unknown
Action detected: unknown
Action detected: unknown
Action detected: unknown

Raw video : bodycam\raw\bodycam_raw_2026-04-30_09-20-11.mp4
Clips     : bodycam\violation
Log       : bodycam\violation\violations_log.csv
